# 3.3.线性回归的简洁实现

在过去的几年里，出于对深度学习强烈的兴趣，许多公司、学者和业余爱好者开发了各种成熟的开源框架。这些框架可以自动化基于梯度的学习算法中重复性的工作。在[<span style="color: #ffbf00; font-weight: bold;"> 3.2节 </span>](./03.02_linear_regression_scratch.ipynb)中，我们只运用了：（1）通过张量来进行数据存储和线性代数；（2）通过自动微分来计算梯度。实际上，由于数据迭代器、损失函数、优化器和神经网络层很常用，现代深度学习库也为我们实现了这些组件。

本节将介绍如何通过使用深度学习框架来简洁地实现[<span style="color: #ffbf00; font-weight: bold;"> 3.2节 </span>](./03.02_linear_regression_scratch.ipynb)中的线性回归模型。

此外，我们还将展示如何使用PyPTO编写自定义算子，并将其封装为标准的nn.Module子类，以便与PyTorch的训练流程无缝集成。

---

## 环境准备

In [ ]:
%pip install pypto==0.2.0 torch torch_npu numpy

In [ ]:
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"

import sys
import numpy as np
import torch
from torch import nn
from torch.utils import data
import pypto

from src.PyPTOLinearFunction import *
from src.PyPTOSquaredLossFunction import *

---

## 3.3.1.生成数据集

与[<span style="color: #ffbf00; font-weight: bold;"> 3.2节 </span>](./03.02_linear_regression_scratch.ipynb)中类似，我们首先生成数据集。

In [4]:
import numpy as np
import torch
from torch.utils import data

In [5]:
def synthetic_data(w, b, num_examples):
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)), device='npu:0')
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape, device='npu:0')
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4], device='npu:0')
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

---

## 3.3.2.读取数据集

我们可以调用框架中现有的API来读取数据。我们将`features`和`labels`作为API的参数传递，并通过数据迭代器指定`batch_size`。此外，布尔值`is_train`表示是否希望数据迭代器对象在每个迭代周期内打乱数据。

In [6]:
def load_array(data_arrays, batch_size, is_train=True):
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [7]:
batch_size = 10
data_iter = load_array((features, labels), batch_size)

使用`data_iter`的方式与我们在[<span style="color: #ffbf00; font-weight: bold;"> 3.2节 </span>](./03.02_linear_regression_scratch.ipynb)中使用`data_iter`函数的方式相同。为了验证是否正常工作，让我们读取并打印第一个小批量样本。与[<span style="color: #ffbf00; font-weight: bold;"> 3.2节 </span>](./03.02_linear_regression_scratch.ipynb)不同，这里我们使用`iter`构造Python迭代器，并使用`next`从迭代器中获取第一项。

In [8]:
next(iter(data_iter))

[tensor([[ 0.7712, -1.0506],
         [ 0.1089,  0.3773],
         [-2.0779,  0.2418],
         [ 1.2379,  0.7059],
         [-1.2974, -1.0420],
         [ 0.1721, -1.0656],
         [-2.1379, -0.5417],
         [-1.4890, -0.4286],
         [-0.4854,  1.2922],
         [ 0.1220,  0.0613]], device='npu:0'),
 tensor([[ 9.2995],
         [ 3.1239],
         [-0.7982],
         [ 4.2719],
         [ 5.1433],
         [ 8.1568],
         [ 1.7628],
         [ 2.6700],
         [-1.1761],
         [ 4.2400]], device='npu:0')]

---

## 3.3.3.定义模型

当我们在[<span style="color: #ffbf00; font-weight: bold;"> 3.2节 </span>](./03.02_linear_regression_scratch.ipynb)中实现线性回归时，我们明确定义了模型参数变量，并编写了计算的代码，这样通过基本的线性代数运算得到输出。但是，如果模型变得更加复杂，且当我们几乎每天都需要实现模型时，自然会想简化这个过程。这种情况类似于为自己的博客从零开始编写网页。做一两次是有益的，但如果每个新博客就需要工程师花一个月的时间重新开始编写网页，那并不高效。

对于标准深度学习模型，我们可以使用框架的预定义好的层。这使我们只需关注使用哪些层来构造模型，而不必关注层的实现细节。我们首先定义一个模型变量`net`，它是一个`Sequential`类的实例。`Sequential`类将多个层串联在一起。当给定输入数据时，`Sequential`实例将数据传入到第一层，然后将第一层的输出作为第二层的输入，以此类推。在下面的例子中，我们的模型只包含一个层，因此实际上不需要`Sequential`。但是由于以后几乎所有的模型都是多层的，在这里使用`Sequential`会让你熟悉"标准的流水线"。

回顾<span style="color: #ffbf00; font-weight: bold;"> 图3.1.2 </span>中的单层网络架构，这一单层被称为*全连接层*（fully-connected layer），因为它的每一个输入都通过矩阵-向量乘法得到它的每个输出。

由于我们运行在 NPU 上，需要自定义算子来实现全连接层。为此，我们首先定义 PyPTO 前向和反向 kernel 函数，然后通过 `torch.autograd.Function` 将其封装为自定义的 `autograd` 函数，最后继承 `nn.Module` 创建 `PyPTOLinear` 层。

当我们定义 `class PyPTOLinear(nn.Module)` 时，我们创建了 `nn.Module` 的子类。 `nn.Module` 是PyTorch中所有神经网络模块的基类，继承它可以获得参数管理、训练/评估模式切换、设备迁移等核心功能。`super().__init__()` 调用父类的初始化方法，这是正确注册模块内部状态的必要步骤。我们将权重和偏置包装为 `nn.Parameter`，这样它们会被自动注册到模块的 `parameters()` 迭代器中，优化器和 hook 机制就能识别并更新它们。`forward` 方法定义了模块的前向计算逻辑：它接收输入X，将其与权重和偏置一同传递给 `PyPTOLinearFunction.apply`，后者调用 PyPTO kernel 在 NPU 上完成实际的矩阵乘法运算。

在原生 PyTorch 中，全连接层在 `nn.Linear` 类中定义，直接传入输入特征数和输出特征数即可。而在 NPU 场景下，我们需要使用前面定义的 `PyPTOLinear` 作为替代——它同样继承自 `nn.Module`，接口与 `nn.Linear` 类似（接受 `in_features` 和 `out_features`），但前向和反向计算由 PyPTO kernel 在 NPU 上执行。

In [9]:
class PyPTOLinear(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(in_features, out_features, device='npu:0'))
        self.bias = nn.Parameter(torch.empty(out_features, device='npu:0'))

    def forward(self, X):
        return PyPTOLinearFunction.apply(X, self.weight, self.bias)

In [10]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(PyPTOLinear(2, 1))
print(net.to("npu"))

Sequential(
  (0): PyPTOLinear()
)


---

## 3.3.4.初始化模型参数

在使用`net`之前，我们需要初始化模型参数。如在线性回归模型中的权重和偏置。深度学习框架通常有预定义的方法来初始化参数。在这里，我们指定每个权重参数应该从均值为0、标准差为0.01的正态分布中随机采样，偏置参数将初始化为零。

正如我们在构造 `PyPTOLinear` 时指定输入和输出尺寸一样，现在我们能直接访问参数以设定它们的初始值。我们通过 `net[0]` 选择网络中的第一个图层（即 `PyPTOLinear` 实例），然后使用 `weight.data` 和 `bias.data` 访问其内部参数。我们可以使用 `normal_` 和 `fill_` 等 in-place 方法来替换参数值。

In [ ]:
# 在这里初始化参数是必须的，因为上文使用 empty 设置参数将不会初始化内存。
net[0].weight.data.normal_(0, 0.01)
print(net[0].bias.data.fill_(0))

tensor([0.], device='npu:0')


---

## 3.3.5.定义损失函数

计算均方误差使用的是 `MSELoss` 类，也称为平方$L_2$范数。默认情况下，它返回所有样本损失的平均值。

类似地，我们也需要为 NPU 上的损失计算编写自定义的 PyPTO kernel，并将其封装为 `nn.Module` 的子类 `PyPTOSquaredLoss`。该类继承自 `nn.Module`，在 `forward` 方法中调用 `PyPTOSquaredLossFunction.apply` 完成NPU上的平方差计算，最后通过 `.mean()` 对 batch 内所有样本的损失取平均，返回标量损失值。

In [12]:
class PyPTOSquaredLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, y_hat, y):
        return PyPTOSquaredLossFunction.apply(y_hat, y).mean()

In [13]:
loss = PyPTOSquaredLoss()

---

## 3.3.6.定义优化算法

小批量随机梯度下降算法是一种优化神经网络的标准工具，PyTorch在`optim`模块中实现了该算法的许多变种。当我们实例化一个`SGD`实例时，我们要指定优化的参数（可通过`net.parameters()`从我们的模型中获得）以及优化算法所需的超参数字典。小批量随机梯度下降只需要设置`lr`值，这里设置为0.03。


In [14]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

---

## 3.3.7.训练

通过深度学习框架的高级API来实现我们的模型只需要相对较少的代码。我们不必单独分配参数、不必定义我们的损失函数，也不必手动实现小批量随机梯度下降。当我们需要更复杂的模型时，高级API的优势将大大增加。当我们有了所有的基本组件，训练过程代码与我们从零开始实现时所做的非常相似。

回顾一下：在每个迭代周期里，我们将完整遍历一次数据集（`train_data`），不停地从中获取一个小批量的输入和相应的标签。对于每一个小批量，我们会进行以下步骤:

* 通过调用`net(X)`生成预测并计算损失`l`（前向传播）。
* 通过进行反向传播来计算梯度。
* 通过调用优化器来更新模型参数。

为了更好的衡量训练效果，我们计算每个迭代周期后的损失，并打印它来监控训练过程。

In [15]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.030714
epoch 2, loss 0.000106
epoch 3, loss 0.000052


下面我们比较生成数据集的真实参数和通过有限数据训练获得的模型参数。要访问参数，我们首先从`net`访问所需的层，然后读取该层的权重和偏置。正如在从零开始实现中一样，我们估计得到的参数与生成数据的真实参数非常接近。

In [16]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([ 4.0364e-04, -2.0981e-05], device='npu:0')
b的估计误差： tensor([0.0002], device='npu:0')


---

## 3.3.8.小结

* 我们可以使用PyTorch的高级API更简洁地实现模型。
* 在PyTorch中，`data`模块提供了数据处理工具，`nn`模块定义了大量的神经网络层和常见损失函数。
* 我们可以通过`_`结尾的方法将参数替换，从而初始化参数。

---



## 3.3.9.练习

1. 如果将小批量的总损失替换为小批量损失的平均值，需要如何更改学习率？
1. 查看深度学习框架文档，它们提供了哪些损失函数和初始化方法？用Huber损失代替原损失，即
    $$l(y,y') = \begin{cases}|y-y'| -\frac{\sigma}{2} & \text{ if } |y-y'| > \sigma \\ \frac{1}{2 \sigma} (y-y')^2 & \text{ 其它情况}\end{cases}\tag{3.3.1}$$
1. 如何访问线性回归的梯度？

参考答案见 [answers/03.03_reference_answer](./answers/03.03_reference_answer.ipynb)